# 11.1.5 샘플 프로그램

앞에서 배운 Chunking, Embedding, Chroma DB 검색, OpenAI 답변 생성을 하나의 흐름으로 연결하는 종합 예제입니다.

실행 흐름:

1. 긴 문서를 chunk로 분할
2. chunk별 embedding 생성
3. Chroma DB에 embedding과 원문 저장
4. 질문을 embedding으로 변환한 뒤 유사 chunk 검색
5. 검색 결과를 OpenAI 모델에 전달해 한국어 답변 생성


## 1. 패키지와 설정

OpenAI API 키는 코드에 직접 쓰지 않고 `.env` 또는 환경변수의 `OPENAI_API_KEY`에서 읽습니다.

Chroma DB 저장 경로는 앞 실습과 충돌하지 않도록 `04_vectordb/chroma_store_1_5`를 사용합니다.

In [1]:
import os
import warnings
from pathlib import Path

import chromadb
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

warnings.filterwarnings("ignore", category=FutureWarning, module="transformers.tokenization_utils_base")

load_dotenv()

CHUNK_SIZE = 256
CHUNK_OVERLAP = 20
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CHROMA_COLLECTION_NAME = "Apt012"

base_dir = Path.cwd()
if base_dir.name != "04_vectordb" and (base_dir / "04_vectordb").exists():
    base_dir = base_dir / "04_vectordb"

PERSIST_DIR = base_dir / "chroma_store_1_5"

print("persist dir:", PERSIST_DIR)

persist dir: c:\Users\Playdata\study\agent-dev\04_vectordb\chroma_store_1_5


## 2. 처리 클래스 정의

원본 txt의 샘플 프로그램처럼 역할별 클래스를 나눠서 작성합니다.

In [2]:
class TextChunkProcessor:
    """긴 텍스트를 관리 가능한 chunk로 분할합니다."""

    def __init__(self, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

    def split_text(self, text):
        documents = self.text_splitter.create_documents([text])
        return [doc.page_content for doc in documents]


class TextEmbedder:
    """SentenceTransformer 모델로 텍스트 embedding을 생성합니다."""

    def __init__(self, model_name=EMBEDDING_MODEL_NAME):
        self.model = SentenceTransformer(model_name)

    def generate_embeddings(self, texts):
        return self.model.encode(texts)


class ChromaDBHandler:
    """Chroma DB 저장과 검색을 담당합니다."""

    def __init__(self, collection_name, persist_dir=PERSIST_DIR):
        self.client = chromadb.PersistentClient(path=str(persist_dir))
        self.collection_name = collection_name
        self.collection = self.client.get_or_create_collection(collection_name)

    def reset_collection(self):
        try:
            self.client.delete_collection(self.collection_name)
        except Exception:
            pass
        self.collection = self.client.get_or_create_collection(self.collection_name)

    def add_to_collection(self, ids, embeddings, metadatas):
        self.collection.add(ids=ids, embeddings=embeddings, metadatas=metadatas)

    def query_collection(self, query_embedding, n_results=2):
        return self.collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=n_results,
            include=["metadatas", "distances"],
        )

## 3. OpenAI 답변 생성 함수

검색된 chunk를 근거로 한국어 답변을 생성합니다. 이 셀은 함수만 정의하고 API 호출은 아직 하지 않습니다.

In [3]:
def get_openai_client():
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY가 설정되어 있지 않습니다. .env 파일 또는 환경변수에 API 키를 넣어주세요.")
    return OpenAI(api_key=api_key)


def generate_chatgpt_response(query, similarity_results, model="gpt-4o-mini"):
    formatted_results = "\n".join(
        f"Result {idx}: {result}" for idx, result in enumerate(similarity_results, start=1)
    )

    prompt = f"""
You are an AI assistant.
Answer the user's question in Korean using only the similarity search results below.

User question:
{query}

Similarity search results:
{formatted_results}

Answer in Korean:
- 핵심 수치를 먼저 말하세요.
- 근거가 된 내용을 간단히 설명하세요.
- 검색 결과에 없는 내용은 추측하지 마세요.
"""

    response = get_openai_client().chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
        max_tokens=500,
        temperature=0.3,
    )

    return response.choices[0].message.content

## 4. 샘플 문서 준비

질문에 답하기 위한 원문 텍스트입니다. 원본 txt의 영어 예시를 정리해 사용합니다.

In [4]:
text = """
The housing business outlook for the metropolitan area and non-metropolitan areas is showing contrasting trends. Housing business operators judged that the housing business in the metropolitan area would improve compared to the previous month, and predicted that the non-metropolitan area would worsen further.

According to the Housing Industry Research Institute's housing business outlook index for housing business operators this month, the national index fell 4.6 points from the previous month to 81.6. The metropolitan area is expected to rise 1.3 points to 107.4, while the non-metropolitan area is expected to show an overall downward trend, falling 5.9 points to 76.0. If the housing business outlook index exceeds the baseline of 100, it means that the proportion of companies that expect the housing business to improve is high, and if it falls below 100, it means the opposite.

Among the metropolitan areas, Gyeonggi showed the largest increase nationwide, rising 8.0 points from 102.5 to 110.5, and Incheon maintained the baseline of 100 for three consecutive months without change. Seoul is expected to fall 4.0 points to 111.9. Amid the ongoing shortage of supply compared to demand, apartment prices in Seoul and its neighboring areas are still on the rise, and expectations for an interest rate cut in the United States are interpreted as having a positive effect on business sentiment.

In the case of non-metropolitan areas, it is expected to fall 5.9 points to 76.0. All metropolitan cities fell, dropping an average of 8.9 points from 87.7 to 78.8, and provincial areas fell an average of 3.6 points from 77.6 to 74.0. It is highly likely that the additional burden on the market will be due to the strengthening of lending such as the Stress DSR Stage 2 regulation and the increase in housing loan interest rates. The slow recovery of housing prices in non-metropolitan areas is interpreted as having a negative impact on the psychology of business owners.

All metropolitan cities fell, with Daejeon showing the largest decline at 17.7 points, from 100.0 to 82.3. It was followed by Daegu at 17.6 points from 95.8 to 78.2, Gwangju at 11.1 points from 66.6 to 55.5, Busan at 4.9 points from 80.9 to 76.0, Ulsan at 1.9 points from 89.4 to 87.5, and Sejong at 0.4 points from 93.7 to 93.3. All provincial regions declined except Gyeongnam, Gyeongbuk, and Chungnam. Daejeon's index appears to have declined due to the prolonged stagnation of the local housing market along with the regional economic downturn.
""".strip()

print("text length:", len(text))

text length: 2530


## 5. Step 1: 문서 Chunking

In [5]:
processor = TextChunkProcessor()
chunks = processor.split_text(text)

print("chunk count:", len(chunks))
for idx, chunk in enumerate(chunks, start=1):
    print(f"Chunk {idx} / length={len(chunk)}")
    print(chunk[:200])
    print("-" * 80)

chunk count: 14
Chunk 1 / length=247
The housing business outlook for the metropolitan area and non-metropolitan areas is showing contrasting trends. Housing business operators judged that the housing business in the metropolitan area wo
--------------------------------------------------------------------------------
Chunk 2 / length=82
previous month, and predicted that the non-metropolitan area would worsen further.
--------------------------------------------------------------------------------
Chunk 3 / length=252
According to the Housing Industry Research Institute's housing business outlook index for housing business operators this month, the national index fell 4.6 points from the previous month to 81.6. The
--------------------------------------------------------------------------------
Chunk 4 / length=253
rise 1.3 points to 107.4, while the non-metropolitan area is expected to show an overall downward trend, falling 5.9 points to 76.0. If the housing business outlook index ex

## 6. Step 2: Embedding 생성

In [6]:
embedder = TextEmbedder()
embeddings = embedder.generate_embeddings(chunks)

print("embedding shape:", embeddings.shape)
print("first embedding length:", len(embeddings[0]))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding shape: (14, 384)
first embedding length: 384


## 7. Step 3: Chroma DB 저장

반복 실행해도 ID 중복 오류가 나지 않도록 기존 컬렉션을 삭제하고 다시 생성합니다.

In [7]:
chroma_handler = ChromaDBHandler(CHROMA_COLLECTION_NAME)
chroma_handler.reset_collection()

chroma_handler.add_to_collection(
    ids=[str(idx) for idx in range(len(chunks))],
    embeddings=[embedding.tolist() for embedding in embeddings],
    metadatas=[{"description": chunk, "chunk_index": idx} for idx, chunk in enumerate(chunks)],
)

print("collection:", CHROMA_COLLECTION_NAME)
print("count:", chroma_handler.collection.count())

collection: Apt012
count: 14


## 8. Step 4: 질문 검색

질문을 embedding으로 바꾼 뒤 Chroma DB에서 가장 가까운 chunk를 검색합니다.

In [8]:
query = "What was the rate of decline in Daejeon's index?"
query_embedding = embedder.generate_embeddings([query])
search_results = chroma_handler.query_collection(query_embedding, n_results=2)

metadata_results = search_results["metadatas"][0]
similarity_results = [result["description"] for result in metadata_results]

print("Query:", query)
print()
for idx, (metadata, distance) in enumerate(
    zip(search_results["metadatas"][0], search_results["distances"][0]),
    start=1,
):
    print(f"Result {idx} / distance={distance}")
    print(metadata["description"])
    print("-" * 80)

Query: What was the rate of decline in Daejeon's index?

Result 1 / distance=0.6517884135246277
to 76.0, Ulsan at 1.9 points from 89.4 to 87.5, and Sejong at 0.4 points from 93.7 to 93.3. All provincial regions declined except Gyeongnam, Gyeongbuk, and Chungnam. Daejeon's index appears to have declined due to the prolonged stagnation of the local
--------------------------------------------------------------------------------
Result 2 / distance=0.6868508458137512
All metropolitan cities fell, with Daejeon showing the largest decline at 17.7 points, from 100.0 to 82.3. It was followed by Daegu at 17.6 points from 95.8 to 78.2, Gwangju at 11.1 points from 66.6 to 55.5, Busan at 4.9 points from 80.9 to 76.0, Ulsan at
--------------------------------------------------------------------------------


## 9. Step 5: OpenAI로 답변 생성

이 셀은 실제 OpenAI API를 호출합니다. `.env`에 `OPENAI_API_KEY`가 있어야 실행됩니다.

In [9]:
answer = generate_chatgpt_response(query, similarity_results)

print("생성된 답변:\n")
print(answer)

생성된 답변:

대전의 지수는 100.0에서 82.3으로 17.7 포인트 하락했습니다. 이는 대전이 모든 대도시 중에서 가장 큰 하락폭을 보였음을 나타냅니다.


## 10. API 호출 없이 결과 확인하기

OpenAI API 호출 전에도 검색 결과만으로 정답 근거가 잘 잡혔는지 확인할 수 있습니다.

In [10]:
for text_chunk in similarity_results:
    if "Daejeon" in text_chunk:
        print(text_chunk)

to 76.0, Ulsan at 1.9 points from 89.4 to 87.5, and Sejong at 0.4 points from 93.7 to 93.3. All provincial regions declined except Gyeongnam, Gyeongbuk, and Chungnam. Daejeon's index appears to have declined due to the prolonged stagnation of the local
All metropolitan cities fell, with Daejeon showing the largest decline at 17.7 points, from 100.0 to 82.3. It was followed by Daegu at 17.6 points from 95.8 to 78.2, Gwangju at 11.1 points from 66.6 to 55.5, Busan at 4.9 points from 80.9 to 76.0, Ulsan at


## 11. 정리

이 샘플은 RAG의 기본 구조를 작게 구현한 예제입니다.

- Chunking으로 긴 문서를 검색 가능한 단위로 나눕니다.
- Embedding으로 chunk와 질문을 벡터로 변환합니다.
- Chroma DB에서 질문과 가까운 chunk를 찾습니다.
- OpenAI 모델은 검색된 chunk를 근거로 최종 답변을 생성합니다.